# **Enron Email Spam Detection: End-to-End NLP Benchmark (Baseline GAP, 1D CNN, GRU/RNN & Transformer)**
### An end-to-end, scalable TensorFlow/Keras natural language processing (NLP) pipeline that builds, trains, evaluates, and compares four distinct deep learning architectures on the **Enron Email Spam Dataset**—complete with threshold tuning, confusion matrix error analysis, and a serializable end-to-end inference pipeline.

---

## Project Highlights & Architecture Progression

### All four models were trained and evaluated on the exact same preprocessed splits (`max_tokens=10000`, `sequence_length=300`, `embedding_dim=128`, `batch_size=128`) using `EarlyStopping` and `ModelCheckpoint` to guarantee a fair, apples-to-apples comparison:

#### 1. **Model 1 — Baseline Bag-of-Embeddings (`Embedding` + `GlobalAveragePooling1D`):** Fast, lightweight baseline averaging word vectors across the email sequence with `mask_zero=True`.
#### 2. **Model 2 — 1D Convolutional Neural Network (`Conv1D` + `GlobalMaxPooling1D`):** Captures local 5-gram phrase patterns (`kernel_size=5`) regularized with `SpatialDropout1D`.
#### 3. **Model 3 — Recurrent Neural Network (`GRU`):** Processes email tokens sequentially to model word order and sentence flow *(Note: Unidirectional `GRU(64)` was used to optimize CPU execution; `Bidirectional(GRU)` with `mask_zero=True` is recommended on GPU hardware)*.
#### 4. **Model 4 — Transformer Encoder (`MultiHeadAttention` + Learned Positional Embeddings):** Compares all 300 tokens simultaneously using multi-head self-attention, residual connections, and `LayerNormalization`.

In [1]:
import sys
import tensorflow as tf


print("Python version:", sys.version)
print("Python executable:", sys.executable)

print("TensorFlow version:", tf.__version__)

Python version: 3.11.15 | packaged by Anaconda, Inc. | (main, Jun 11 2026, 15:12:53) [MSC v.1942 64 bit (AMD64)]
Python executable: C:\Users\HP\miniconda3\envs\dog-vision\python.exe
TensorFlow version: 2.21.0


In [2]:
from datasets import load_dataset


import datasets

print("Datasets library version:", datasets.__version__)

# Load the real-world Enron-Spam dataset (~33,700 emails)
email_dataset = load_dataset("SetFit/enron_spam")

# Inspect the structure and column names
print(email_dataset)

C:\Users\HP\miniconda3\envs\dog-vision\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Datasets library version: 5.1.0


Repo card metadata block was not found. Setting CardData to empty.


DatasetDict({
    train: Dataset({
        features: ['message_id', 'text', 'label', 'label_text', 'subject', 'message', 'date'],
        num_rows: 31716
    })
    test: Dataset({
        features: ['message_id', 'text', 'label', 'label_text', 'subject', 'message', 'date'],
        num_rows: 2000
    })
})


### **Now we need to do 1, 2, 3**
#### **Below we use `label_text` and not `label` bcos the Enron-Spam dataset contains two different columns that represent the email labels in different formats. i.e:**
#### **label:** contains numerical labels (typically 0 and 1).
#### **label_text:** contains human-readable labels (ham and spam).
#### Hence we use `label_text` and not the `label` to correct `ham` to `not_spam`

#### **1) Verifying labels**

In [3]:
email_dataset["train"].to_pandas()["label_text"].value_counts()

label_text
spam    16163
ham     15553
Name: count, dtype: int64

### **OR no need to convert to pandas to see the labels**

In [4]:
from collections import Counter

Counter(email_dataset["train"]["label_text"])

Counter({'spam': 16163, 'ham': 15553})

#### **2) First 10 original labels**

In [5]:
email_dataset["train"]["label_text"][:10]

['spam', 'ham', 'spam', 'ham', 'spam', 'spam', 'spam', 'spam', 'ham', 'spam']

#### **3) Replacing `ham` with `not_spam`**

In [6]:
# Convert splits to pandas DataFrames
train_data = email_dataset["train"].to_pandas()
test_data = email_dataset["test"].to_pandas()

# Combine subject and message into a single 'text' column, and rename 'ham' to 'not_spam'
for df in [train_data, test_data]:
    df["text"] = df["subject"].fillna("") + " " + df["message"].fillna("")
    df["label"] = df["label_text"].replace({"ham": "not_spam"})

# Keep only the two columns your pipeline expects
train_data = train_data[["text", "label"]]
test_data = test_data[["text", "label"]]

print("Training shape:", train_data.shape)
print("Test shape:", test_data.shape)
print(train_data.head())

Training shape: (31716, 2)
Test shape: (2000, 2)
                                                text     label
0  any software just for 15 $ - 99 $ understandin...      spam
1  perspective on ferc regulatory action client c...  not_spam
2  wanted to try ci 4 lis but thought it was way ...      spam
3  enron / hpl actuals for december 11 , 2000 tec...  not_spam
4  looking for cheap high - quality software ? ro...      spam


# **A) Duplicate detection and removal**

In [7]:
import pandas as pd

train_texts = train_data["text"]
train_texts[:5]

# Convert train_texts in Hugging face column to pandas series
train_texts = pd.Series(train_texts)
train_texts.head()

0    any software just for 15 $ - 99 $ understandin...
1    perspective on ferc regulatory action client c...
2    wanted to try ci 4 lis but thought it was way ...
3    enron / hpl actuals for december 11 , 2000 tec...
4    looking for cheap high - quality software ? ro...
Name: text, dtype: str

### **a) Removing Normal Duplicates: These are text that appear more than once**
#### **1) Count the number of duplicate emails**

In [8]:
duplicate_count = train_texts.duplicated().sum
print("Number of duplicate emails: ", duplicate_count())

Number of duplicate emails:  2874


#### **2) Check if any exact duplicates have conflicting labels (fast aggregation, safe at scale)**

In [9]:
conflicts = train_data.groupby("text")["label"].nunique()
print("Duplicate texts with conflicting labels: ", (conflicts > 1).sum())

Duplicate texts with conflicting labels:  0


#### **3) (Optional) Drop any conflicting texts completely so you don't train on contradictory labels**

In [10]:
conflicting_texts = conflicts[conflicts > 1].index
train_data_no_conflicts = train_data[~train_data["text"].isin(conflicting_texts)]

#### **4) Remove exact duplicate emails and reset the index**

In [11]:
train_data_cleaned = train_data_no_conflicts.drop_duplicates(
    subset=["text"],
    keep="first"
).reset_index(drop=True)

# Verify cleanup results
removed_count = len(train_data) - len(train_data_cleaned)
print(f"Original dataset size: {len(train_data)}")
print(f"Cleaned dataset size:  {len(train_data_cleaned)} (Removed {removed_count} rows)")
print("Duplicates remaining: ", train_data_cleaned["text"].duplicated().sum())

Original dataset size: 31716
Cleaned dataset size:  28842 (Removed 2874 rows)
Duplicates remaining:  0


#### **5) Another duplicates we need to investigate are those in the test dataset**
#### **Clean Test Data & Eliminate Train-to-Test Data Leakage**
##### **i) Quick audit before cleaning (uses fast hash lookups, safe for millions of rows)**

In [12]:
print("Duplicate emails inside test data:", test_data["text"].duplicated().sum())
print("Exact overlap between train and test:", test_data["text"].isin(train_data_cleaned["text"]).sum())

Duplicate emails inside test data: 18
Exact overlap between train and test: 333


#### **ii) (Optional - Course Benchmark Only) Remove exact duplicates within test data.**
#### **Note:** In real-world production, skip this line so test data keeps natural frequency. And this is bcos your test data represents the real world, where identical emails naturally happen multiple times.

In [13]:
test_data_cleaned = test_data.drop_duplicates(subset=["text"], keep="first")
test_data_cleaned.head()

,text,label
0,expande tu imagen ! ! ! ! ! ! ! ! ! si no pued...,spam
1,paliourg learning for life enlarge your member...,spam
2,"cure premature ejaculation hello ,\ndid you ej...",spam
3,re : noms / actual flow for 3 / 19 / 01 we agr...,not_spam
4,ehronline web address change this message is i...,not_spam


#### **iii) Remove any test emails that already exist in train_data_cleaned (Prevents Data Leakage)**

In [14]:
test_data_cleaned = test_data_cleaned[
    ~test_data_cleaned["text"].isin(train_data_cleaned["text"])
].reset_index(drop=True)

#### **iv) Final verification checks**

In [15]:
print("\nOriginal test dataset size:", len(test_data))
print("Cleaned test dataset size: ", len(test_data_cleaned))
print("Duplicates remaining in test:", test_data_cleaned["text"].duplicated().sum())
print("Remaining train-test overlap:", test_data_cleaned["text"].isin(train_data_cleaned["text"]).sum())


Original test dataset size: 2000
Cleaned test dataset size:  1652
Duplicates remaining in test: 0
Remaining train-test overlap: 0


## **b) Removing near-duplicates for `training data` (Scalable & Bug-Free)**

In [16]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

#### **1) Convert training emails into TF-IDF character n-grams**
#### max_features=20000 and float32 prevent memory bloat and speed up cosine math on large datasets

In [17]:
tfidf_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    min_df=2,
    max_features=20000,
    dtype=np.float32
)

train_tfidf = tfidf_vectorizer.fit_transform(train_data_cleaned["text"])
print("TF-IDF matrix shape:", train_tfidf.shape)

TF-IDF matrix shape: (28842, 20000)


#### **2) Find nearest neighbors using multi-core cosine similarity (n_jobs=-1 uses all CPU cores)**

In [18]:
nearest_neighbors = NearestNeighbors(
    n_neighbors=3,
    metric="cosine",
    algorithm="brute",
    n_jobs=-1
)

nearest_neighbors.fit(train_tfidf)
distances, indices = nearest_neighbors.kneighbors(train_tfidf)

#### **3) Extract candidate pairs above our 0.95 automated threshold using fast NumPy arrays**

In [19]:
THRESHOLD = 0.95
similarities = 1 - distances[:, 1:]        # Skip column 0 (the email itself)
neighbor_indices = indices[:, 1:]

# Find row/col positions where similarity >= THRESHOLD
rows, cols = np.where(similarities >= THRESHOLD)

# Build DataFrame directly from NumPy arrays (much faster than appending in Python loops)
near_duplicates_df = pd.DataFrame({
    "email_1": rows,
    "email_2": neighbor_indices[rows, cols],
    "similarity": similarities[rows, cols]
})

#### **4) Remove reverse duplicate pairs ((A, B) vs (B, A)) using vectorized NumPy min/max**

In [20]:
e1 = np.minimum(near_duplicates_df["email_1"], near_duplicates_df["email_2"])
e2 = np.maximum(near_duplicates_df["email_1"], near_duplicates_df["email_2"])
near_duplicates_df["email_1"] = e1
near_duplicates_df["email_2"] = e2

near_duplicates_df = (
    near_duplicates_df
    .sort_values(by="similarity", ascending=False)
    .drop_duplicates(subset=["email_1", "email_2"])
    .reset_index(drop=True)
)

print("Unique near-duplicate pairs (>= 0.95):", len(near_duplicates_df))


Unique near-duplicate pairs (>= 0.95): 6496


#### **5) Safety Check: Verify labels match BEFORE dropping any rows**

In [21]:
label_1 = train_data_cleaned["label"].iloc[near_duplicates_df["email_1"]].to_numpy()
label_2 = train_data_cleaned["label"].iloc[near_duplicates_df["email_2"]].to_numpy()

safe_to_drop = near_duplicates_df[label_1 == label_2]
indices_to_drop = safe_to_drop["email_2"].unique()

print("Pairs with matching labels (safe to drop):", len(safe_to_drop))

Pairs with matching labels (safe to drop): 6458


#### **6) Drop all verified near-duplicates in one bulk operation and reset index ONCE at the end**

In [22]:
train_data_cleaned = train_data_cleaned.drop(index=indices_to_drop).reset_index(drop=True)

print(f"Automatically removed {len(indices_to_drop)} near-duplicate emails.")
print("Final training dataset shape:", train_data_cleaned.shape)

Automatically removed 3944 near-duplicate emails.
Final training dataset shape: (24898, 2)


### **NOTE:**
### **`0.90` is a great threshold for checking `train-to-test data leakage`, while `0.95` is safer when `automatically deleting duplicates inside your training data`.**

## **c) Removing near-duplicates for `test data`**
#### ***Convert test emails into TF-IDF features***
#### We'll reuse the tfidf_vectorizer we fitted on the training data. This is important because we want both datasets represented using the same vocabulary.
#### So i will skip step 2 & 3 as seen in **`practcing-sm-parts-of-email-spam-identification.ipynb` bcos leaving natural near-duplicates inside test_data_cleaned keeps your final accuracy score realistic.**

In [23]:
test_tfidf = tfidf_vectorizer.transform(
    # transform() converts the test emails into TF-IDF features.
    # We use transform(), not fit_transform(), because the vectorizer has already learned 
    # its character n-gram vocabulary from the training data.
    test_data_cleaned["text"]
)

print("TF-IDF matrix shape:", test_tfidf.shape)

TF-IDF matrix shape: (1652, 20000)


## **d) Removing near-duplicates between the training and test datasets.**
#### This is important because similar emails appearing in both datasets could affect our model evaluation.

#### **Refresh train_tfidf so its rows match the deduplicated train_data_cleaned**

In [24]:
train_tfidf = tfidf_vectorizer.transform(train_data_cleaned["text"])

#### **1) Find the nearest training emails for each test email (using all CPU cores)**

In [25]:
cross_neighbors = NearestNeighbors(
    n_neighbors=6,
    metric="cosine",
    algorithm="brute",
    n_jobs=-1
)

cross_neighbors.fit(train_tfidf)
cross_distances, cross_indices = cross_neighbors.kneighbors(test_tfidf)

print("Distances shape:", cross_distances.shape)
print("Indices shape:", cross_indices.shape)

Distances shape: (1652, 6)
Indices shape: (1652, 6)


#### **2) Identify potential near-duplicates across both datasets using 0.90 threshold**

In [26]:
similarity_threshold = 0.90
cross_near_duplicates = []

for i in range(len(test_data_cleaned)):
    for j in range(cross_distances.shape[1]):
        similarity = 1 - cross_distances[i, j]
        if similarity >= similarity_threshold:
            cross_near_duplicates.append((
                i,
                cross_indices[i, j],
                similarity
            ))

print("Potential near-duplicate pairs:", len(cross_near_duplicates))

Potential near-duplicate pairs: 1504


#### **3) Store candidate pairs in a clean DataFrame**

In [27]:
cross_near_duplicates_df = pd.DataFrame(
    cross_near_duplicates,
    columns=["test_email", "train_email", "similarity"]
).sort_values(by="similarity", ascending=False).reset_index(drop=True)

#### **4) Remove leaked near-duplicates from test data only**

In [28]:
test_indices_to_drop = cross_near_duplicates_df["test_email"].astype(int).unique()
test_data_cleaned = test_data_cleaned.drop(index=test_indices_to_drop).reset_index(drop=True)

print(f"Removed {len(test_indices_to_drop)} leaked test emails.")
print("Updated test dataset shape:", test_data_cleaned.shape)

Removed 492 leaked test emails.
Updated test dataset shape: (1160, 2)


## **B) Text cleaning and normalization**
#### **The next step is checking for missing or empty email messages before we begin text cleaning and normalization.**
#### **We want to know whether any email contains:**
#### - A missing (NaN) text value.
#### - An empty string ("").
#### - Only whitespace (spaces, tabs, or newlines).
#### **Let's start with the training dataset.**

#### ***1) Check for missing email text***
#### Below is a fc that checks for missing email text

In [29]:
#  Step A: Define the detector function to count missing or whitespace-only emails
def check_email_texts(data, dataset_name):
    """
    Checks for missing, empty, and whitespace-only email messages.
    """
    # missing: counts missing text values (NaN).
    missing = data["text"].isna().sum()

    # fillna(""): temporarily replaces missing values with empty strings for checking.
    # astype(str): ensures non-string types (like numbers) don't crash .str.strip().
    empty = (
        # str.strip(): removes surrounding whitespace for the purpose of the check
        data["text"].fillna("").astype(str).str.strip() == ""
    ).sum()
    # == "": identifies empty or whitespace-only messages.
    
    print(f"{dataset_name} dataset:")
    print("Missing emails:", missing)
    print("Empty or whitespace-only emails:", empty)

# Step B: Run the detector BEFORE cleaning to see what needs to be removed
print("--- Before Removing Empty Emails ---")
check_email_texts(train_data_cleaned, "Training")
check_email_texts(test_data_cleaned, "Test")

# Step C: Remove any missing, empty, or whitespace-only rows from both datasets
train_data_cleaned = train_data_cleaned[
    train_data_cleaned["text"].fillna("").astype(str).str.strip() != ""
].reset_index(drop=True)

test_data_cleaned = test_data_cleaned[
    test_data_cleaned["text"].fillna("").astype(str).str.strip() != ""
].reset_index(drop=True)

# Step D: Run the detector AGAIN to confirm both datasets now have 0 empty emails
print("\n--- After Removing Empty Emails ---")
check_email_texts(train_data_cleaned, "Training")
check_email_texts(test_data_cleaned, "Test")

--- Before Removing Empty Emails ---
Training dataset:
Missing emails: 0
Empty or whitespace-only emails: 1
Test dataset:
Missing emails: 0
Empty or whitespace-only emails: 0

--- After Removing Empty Emails ---
Training dataset:
Missing emails: 0
Empty or whitespace-only emails: 0
Test dataset:
Missing emails: 0
Empty or whitespace-only emails: 0


#### ***2) Inspect the email text before cleaning***
#### **Before writing a cleaning function, we need to understand what appears in the emails. We should inspect:**
#### 1. HTML tags.
#### 2. URLs and links.
#### 3. Email addresses.
#### 4. Special characters and punctuation.
#### 5. Numbers and currency symbols.
#### We shouldn't automatically remove these because some of them may contain useful spam indicators.
#### **Let's create a reusable function to display sample emails.**
#### **i) Create the function**
##### Include this step. Even when working with millions of rows, inspecting a fixed, small number of samples (n=5) is an essential sanity check to catch weird formatting (like HTML entities or links) before writing cleaning rules.

In [30]:
def display_email_samples(data, dataset_name, n=5):
    """
    Displays sample email messages for inspection.
    """
    print(f"{dataset_name} email samples:\n")

    for i in range(n):
        print(f"Email {i + 1}:")
        print(data["text"].iloc[i])
        print("-" * 80)

#### **ii) Inspect both datasets**

In [31]:
display_email_samples(train_data_cleaned, "Training")
display_email_samples(test_data_cleaned, "Test")

Training email samples:

Email 1:
any software just for 15 $ - 99 $ understanding oem software
lead me not into temptation ; i can find the way myself .
# 3533 . the law disregards trifles .
--------------------------------------------------------------------------------
Email 2:
perspective on ferc regulatory action client conf call today , jun e 19 th , 2 : 00 pm edt
perspective on ferc regulatory action client conference call
today , tuesday , june 19 th
2 : 00 pm edt
host : ray niles , power / natural gas analyst
speaker : steve bergstrom , president & coo of dynegy
steve bergstrom , president and chief operating officer of dynegy , will join
us at 2 : 00 p . m . today for a conference call discussion of the recent ferc
action imposing price controls in the west . the discussion will be followed
by q & a .
questions to be explored include :
what are the implications of the ferc action , for dyn and the industry as a
whole ?
what is the earnings impact ?
what are the risks of furthe

#### **iii) Building our reusable text-cleaning function.**
#### So in **`going-thru-some-parts-of-email-spam-project.ipynb`** this i did not repeat the steps done from **`a) to g)`**. This is bcos it was being explained so oit was done in difft steps. But now we can just do the whole in one step which is the right way and is efficient way to go when performing the steps **`a) to g)`**.

#### **a) Create a fc that will do the below cleaning**
#### **How Your Tutor's Steps Map to That Single Function**
##### Sub-step a (import html): Became html.unescape(str(text)) at the start of the function.
##### Sub-step b (Markdown links): Became MD_LINK_RE.sub(r"\1", text) (plus BLOCKQUOTE_RE to strip > quote symbols).
##### Sub-step c (Standalone URLs): Became URL_RE.sub(" URL ", text).
##### Sub-step d (Email addresses): Became EMAIL_RE.sub(" EMAIL ", text).
##### Sub-step e (Lowercase & whitespace): Became text = text.lower() and WHITESPACE_RE.sub(" ", text).strip().
##### Sub-step f (Escaped backslashes): Became text = text.replace("\\", "").

In [32]:
import html
import re

# Compile regex patterns once for fast execution on large datasets
BLOCKQUOTE_RE = re.compile(r"(?m)^\s*>\s?")
MD_LINK_RE = re.compile(r"\[([^\]]+)\]\([^)]+\)")
URL_RE = re.compile(r"https?://\S+")
EMAIL_RE = re.compile(r"\b[\w.+-]+@[\w.-]+\.[a-zA-Z]{2,}\b")
WHITESPACE_RE = re.compile(r"\s+")

def clean_email(text):
    """
    Cleans and normalizes email text efficiently for large datasets.
    """
    # Ensure input is a string in case of unexpected data types
    text = html.unescape(str(text))

    # Remove Markdown blockquote markers and link formatting
    text = BLOCKQUOTE_RE.sub("", text)
    text = MD_LINK_RE.sub(r"\1", text)

    # Lowercase first so special tokens stay uppercase
    text = text.lower()

    # Replace URLs and email addresses with distinct uppercase tokens
    text = URL_RE.sub(" URL ", text)
    text = EMAIL_RE.sub(" EMAIL ", text)

    # Remove escape backslashes and normalize whitespace
    text = text.replace("\\", "")
    text = WHITESPACE_RE.sub(" ", text).strip()

    return text

#### **b) Apply the function to both datasets**
#### We're creating a new column called clean_text rather than overwriting text. This allows us to compare the original emails with their cleaned versions.

In [33]:
# Clean training email texts
train_data_cleaned["clean_text"] = train_data_cleaned["text"].apply(clean_email)

# Clean test email texts
test_data_cleaned["clean_text"] = test_data_cleaned["text"].apply(clean_email)

#### **c) Compare original and cleaned emails**
#### Let's inspect a few examples to ensure the cleaning function worked correctly on real dataset messages, not just our sample.

In [34]:
def compare_cleaned_emails(data, dataset_name, n=5):
    """
    Displays original and cleaned email texts side by side.
    """

    print(f"{dataset_name} dataset:\n")

    for i in range(n):
        print(f"Email {i + 1}")
        print("Original:")
        print(data["text"].iloc[i])

        print("\nCleaned:")
        print(data["clean_text"].iloc[i])

        print("-" * 80)

#### **Use it:**

In [35]:
compare_cleaned_emails(train_data_cleaned, "Training")
compare_cleaned_emails(test_data_cleaned, "Test")

Training dataset:

Email 1
Original:
any software just for 15 $ - 99 $ understanding oem software
lead me not into temptation ; i can find the way myself .
# 3533 . the law disregards trifles .

Cleaned:
any software just for 15 $ - 99 $ understanding oem software lead me not into temptation ; i can find the way myself . # 3533 . the law disregards trifles .
--------------------------------------------------------------------------------
Email 2
Original:
perspective on ferc regulatory action client conf call today , jun e 19 th , 2 : 00 pm edt
perspective on ferc regulatory action client conference call
today , tuesday , june 19 th
2 : 00 pm edt
host : ray niles , power / natural gas analyst
speaker : steve bergstrom , president & coo of dynegy
steve bergstrom , president and chief operating officer of dynegy , will join
us at 2 : 00 p . m . today for a conference call discussion of the recent ferc
action imposing price controls in the west . the discussion will be followed
by q & a .

## **C) Splitting training and validation data.**

In [36]:
from sklearn.model_selection import train_test_split

def split_data(data, test_size=0.2, random_state=42):
    """
    Splits email data into training and validation sets.
    """

    # Separate features and labels
    X = data["clean_text"]
    y = data["label"]

    # Split the data
    X_train, X_valid, y_train, y_valid = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )

    return X_train, X_valid, y_train, y_valid

### **Use the fc**

In [37]:
X_train, X_valid, y_train, y_valid = split_data(train_data_cleaned)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_valid))
print("\nTraining label distribution:\n", y_train.value_counts(normalize=True))
print("\nValidation label distribution:\n", y_valid.value_counts(normalize=True))

Training samples: 19917
Validation samples: 4980

Training label distribution:
 label
not_spam    0.529397
spam        0.470603
Name: proportion, dtype: float64

Validation label distribution:
 label
not_spam    0.529518
spam        0.470482
Name: proportion, dtype: float64


## **D) Tokenization**
#### **TensorFlow's TextVectorization layer** is what we will use for tokenization and converting our email text into numerical sequences.
#### **It performs several related tasks:**
#### 1. Tokenization: Splits each email into individual tokens (usually words).
#### 2. Vocabulary creation: Builds a collection of unique tokens from the training emails.
#### 3. Integer encoding: Assigns each token a numerical ID.
#### For example:
#### **Original email:**
#### "Get your prize now"

#### **Tokenized:**
#### ["get", "your", "prize", "now"]

#### **Integer encoded:**
#### [15, 8, 42, 6]

#### ***1) Create the TextVectorization layer***
#### We set max_tokens=10000 as an upper limit, not a requirement to use exactly 10,000 tokens.

In [38]:
# Create the layer
text_vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=10000, # max_tokens=10000: Limits the vocabulary to a maximum of 10,000 tokens
    # of most frequent words prevents rare typos, random hashes, or spammer gibberish from 
    # bloating your model's embedding matrix..
    output_mode="int", # output_mode="int": Converts tokens into integer IDs.
    standardize=None
    # standardize=None: Preserves our existing text formatting, including punctuation, rather 
    # than automatically stripping punctuation.
)

#### ***Step 2: Adapt the TextVectorization layer***
#### **What does .adapt() do?**
#### **It examines the training emails and:**
#### 1. Identifies the individual tokens (words).
#### 2. Builds a vocabulary of the tokens it finds.
#### 3. Assigns each token an integer ID.
#### 4. Keeps the most frequent tokens, subject to our max_tokens=10000 limit.
#### **For example, if the word `free` appears frequently, it will receive an `integer ID` in the vocabulary.**

In [39]:
text_vectorizer.adapt(X_train)
# Passing only X_train into .adapt() strictly prevents data leakage from your validation or
# test sets.

## **E) Vocabulary creation**
#### ***1) Retrieve the vocabulary***
#### I used only this step here bcos when working with large datasets 2, 3, 4, 5 as i did in **`going-thru-some-parts-of-email-spam-project.ipynb`**

In [40]:
vocabulary = text_vectorizer.get_vocabulary()

print("Vocabulary size:", len(vocabulary))
print("\nFirst 20 tokens:")
print(vocabulary[:20])

Vocabulary size: 10000

First 20 tokens:
['', '[UNK]', np.str_('.'), np.str_('-'), np.str_(','), np.str_('the'), np.str_('to'), np.str_('/'), np.str_(':'), np.str_('and'), np.str_('of'), np.str_('a'), np.str_('in'), np.str_('for'), np.str_("'"), np.str_('you'), np.str_('is'), np.str_('i'), np.str_('enron'), np.str_('this')]


#### **What are we doing?**
#### - get_vocabulary() retrieves the words and tokens learned by the vectorizer.
#### - len(vocabulary) tells us how many tokens are in the vocabulary.
#### - vocabulary[:20] displays the first 20 tokens.
#### **One important detail: the first two entries are reserved for special purposes:**
#### - "" — padding.
#### - "[UNK]" — tokens that are not in the vocabulary (unknown words).
#### **The remaining tokens are arranged by frequency, from most frequent to less frequent.**

## **F) Numerical conversion (Setting up the Fixed-Length Vectorizer)**
#### When dealing with a big dataset, these are the main steps to follow

#### ***1) Splits each training email into words and counts how many words are in each email.***

In [41]:
train_lengths = X_train.str.split().str.len()

# Prints the maximum, average, and 95th percentile word counts so we can pick a smart sequence length.
print("Longest training email:", train_lengths.max())
print("Average training email length:", round(train_lengths.mean(), 2))
print("95th percentile:", train_lengths.quantile(0.95))

Longest training email: 34912
Average training email length: 291.41
95th percentile: 843.2000000000007


#### ***2) Defines a reusable function to build and train our text-to-number converter.***
#### **Now what we set below is dependent on `Average training email length: 291.41`, `95th percentile: 843.2000000000007`**
#### **Why 300?**
#### - It captures the full average Enron email without making your laptop CPU slow down like 850 would.
#### - Because spam clues and corporate headers usually appear in the first 300 words, 300 gives you a massive accuracy boost over 150 while staying fast to train.

In [42]:
def create_text_vectorizer(train_texts, max_tokens=10000, sequence_length=300):
    """
    Creates and adapts a fixed-length TextVectorization layer.
    """
    # Initializes TensorFlow's TextVectorization layer.
    vectorizer = tf.keras.layers.TextVectorization(
        max_tokens=max_tokens,# Caps the vocabulary at the 10,000 most frequent tokens.
        output_mode="int", # Outputs whole numbers (integer IDs) instead of raw text.
        output_sequence_length=sequence_length, 
        # Pads short emails with 0s and cuts long emails at 150 tokens.
        standardize=None # Skips built-in text cleaning since we already cleaned the text.
    )
    
    # Scans ONLY the training emails to build the vocabulary and assign integer IDs.
    vectorizer.adapt(train_texts)
    
    # Returns the ready-to-use vectorizer.
    return vectorizer

#### ***3) Builds our vectorizer using X_train, capping vocabulary at 10,000 and sequence length at 300 or 400.***

In [43]:
text_vectorizer = create_text_vectorizer(X_train, max_tokens=10000, sequence_length=300)

# Sanity check: Converts just the very first training email to verify words become numbers properly.
print("\nOriginal sample email:\n", X_train.iloc[0])
print("\nVectorized sample email:\n", text_vectorizer(X_train.iloc[:1])[0].numpy())


Original sample email:
 major news released after the close , a must watch tomorrow stock hits 52 wk high . . . strong buy issued the players network , inc . otc bulletin board : symbol : pntv current price : $ 1 . 65 up . 17 cents , or 40 % in the last 4 days of trading , we see $ 3 . 50 around the corner 5 day projection : $ 3 . 50 press release source : players network players network ( pntv ) media and entertainment executive andrew orgel to appear on ' corporate strategies with tim connolly ' sunday , july 17 th at 10 pm et las vegas , july 15 / prnewswire - firstcall / - - andrew orgel , media and entertainment executive of players network will appear live sunday night , july 17 th at 10 pm et on " corporate strategies with tim connolly " . players network ( otc bulletin board : pntv - news ) is the leading media company in the gaming lifestyle category . pntv partners with major gaming industry companies in the marketing and promotion of their brands through the development of 

## **G) Padding and Truncation (Transforming All Datasets to Uniform Shapes)**
#### ***1) Converting all trainings and validation emails to uniform matrix***

In [44]:
# Converts all training emails into a uniform matrix of shape (num_train_emails, 300).
X_train_vectorized = text_vectorizer(X_train)

# Converts all validation emails using the exact same rules and vocabulary.
X_valid_vectorized = text_vectorizer(X_valid)

# Prints the final dimensions of all three datasets to confirm every row has length 300.
print("\nTraining shape:", X_train_vectorized.shape)
print("Validation shape:", X_valid_vectorized.shape)


Training shape: (19917, 300)
Validation shape: (4980, 300)


#### ***2) Converting test emails to uniform matrix***

In [45]:
# Extracts the cleaned text and labels from our final test dataset.
X_test = test_data_cleaned["clean_text"]
y_test = test_data_cleaned["label"]

# Converts all test emails into a uniform matrix of shape (num_test_emails, 300).
X_test_vectorized = text_vectorizer(X_test)

# Prints the final dimensions of all three datasets to confirm every row has length 300.
print("Test shape:", X_test_vectorized.shape)

Test shape: (1160, 300)


## **H) Preparing our labels OR Label encoding.**
#### ***1) Inspect class balance across all three splits (fast even on millions of rows).***

In [46]:
print("Training label distribution:\n", y_train.value_counts())
print("\nValidation label distribution:\n", y_valid.value_counts())
print("\nTest label distribution:\n", y_test.value_counts())

Training label distribution:
 label
not_spam    10544
spam         9373
Name: count, dtype: int64

Validation label distribution:
 label
not_spam    2637
spam        2343
Name: count, dtype: int64

Test label distribution:
 label
not_spam    657
spam        503
Name: count, dtype: int64


#### ***2) Define a reusable function to convert text labels into binary integers (0 and 1).***
#### **We'll convert:**
#### - not_spam → 0
#### - spam → 1
#### **Why use these values?** They give us a binary classification target that our neural network can learn to predict.

In [47]:
def encode_labels(labels):
    """
    Converts 'not_spam' to 0 and 'spam' to 1, and ensures no unmapped NaN values exist.
    """
    # Maps text categories to integers.
    encoded = labels.map({"not_spam": 0, "spam": 1})
    
    # Safety check for large datasets: stops immediately if any unexpected label turned into NaN.
    assert encoded.isna().sum() == 0, "Error: Found unexpected label values that became NaN!"
    
    # Returns the clean integer Series.
    return encoded.astype("int32")

#### ***3) Apply the encoding function to training, validation, and test labels.***
#### **The below converts the text labels into numerical values:**
#### - not_spam → 0
#### - spam → 1
#### We need numerical labels because the neural network will learn to distinguish between the two classes using these values.

In [48]:
y_train_encoded = encode_labels(y_train)
y_valid_encoded = encode_labels(y_valid)
y_test_encoded = encode_labels(y_test)

#### ***4) Convert pandas Series into fast NumPy arrays for TensorFlow model training.***

In [49]:
y_train_encoded = y_train_encoded.to_numpy()
y_valid_encoded = y_valid_encoded.to_numpy()
y_test_encoded = y_test_encoded.to_numpy()

# Verify final shapes match between email matrices (X) and target labels (y).
print("\nFinal Shapes Verification:")
print(f"Training:   X={X_train_vectorized.shape}, y={y_train_encoded.shape}, dtype={y_train_encoded.dtype}")
print(f"Validation: X={X_valid_vectorized.shape}, y={y_valid_encoded.shape}, dtype={y_valid_encoded.dtype}")
print(f"Test:       X={X_test_vectorized.shape}, y={y_test_encoded.shape}, dtype={y_test_encoded.dtype}")


Final Shapes Verification:
Training:   X=(19917, 300), y=(19917,), dtype=int32
Validation: X=(4980, 300), y=(4980,), dtype=int32
Test:       X=(1160, 300), y=(1160,), dtype=int32


# **Build, Summarize, and Compile the Email Classification Model**
#### This is the code for the above part

#### ***1) Define the model architecture***

In [50]:
email_model = tf.keras.Sequential([
    # Embedding: Turns each of the 10,000 word IDs into a dense 128-number vector.
    # mask_zero=True tells later layers to ignore padding zeros (0) when averaging.
    tf.keras.layers.Embedding(
        input_dim=10000,
        output_dim=128,
        mask_zero=True
    ),

    # GlobalAveragePooling1D: Averages the word vectors across the email into one summary vector.
    tf.keras.layers.GlobalAveragePooling1D(),

    # Dense(64): Hidden layer with 64 neurons to learn combinations of spam patterns.
    tf.keras.layers.Dense(64, activation="relu"),

    # Dense(1): Output layer producing a single probability between 0 (not_spam) and 1 (spam).
    tf.keras.layers.Dense(1, activation="sigmoid")
])

#### ***2) Display the initial model summary***
#### This helps us understand the model's structure and how many trainable parameters it contains before training

In [51]:
email_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling1d             │ ?                           │               0 │
│ (GlobalAveragePooling1D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

#### ***3) Build the model and inspect the summary***

In [52]:
# Explicitly sets the input shape to (batch_size, 300 tokens) and prints layer parameters.
email_model.build(input_shape=(None, 300))
email_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ (None, 300, 128)            │       1,280,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling1d             │ (None, 128)                 │               0 │
│ (GlobalAveragePooling1D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │              65 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,288,321 (4.91 MB)

 Trainable params: 1,288,321 (4.91 MB)

 Non-trainable params: 0 (0.00 B)

#### ***4) Compile the model***

In [53]:
# Attaches the loss function, optimizer, and evaluation metrics to prepare for training.
email_model.compile(
    loss=tf.keras.losses.BinaryCrossentropy(),      # Standard loss for binary (0 vs 1) classification.
    optimizer=tf.keras.optimizers.Adam(),           # Adaptive optimizer that updates weights efficiently.
    metrics=[
        "accuracy",                                 # Overall percentage of correct predictions.
        tf.keras.metrics.Precision(name="precision"), # Out of emails flagged as spam, how many actually were?
        tf.keras.metrics.Recall(name="recall")        # Out of all actual spam emails, how many did we catch?
    ]
)

# **Training the Email Spam Identification Model**
#### Now we move to training. This is where the model learns patterns from our training emails and checks its performance against the validation emails.
## **Train, Save, and Verify the Email Spam Identification Model**

#### ***1) Train the model with EarlyStopping and ModelCheckpoint***

In [54]:
# Uses larger batch_size (128) for faster execution on large datasets.
history = email_model.fit(
    X_train_vectorized,
    y_train_encoded,
    epochs=10,
    batch_size=128, # I increased batch_size from 32 to 128 because your goal is building a
    # pipeline ready for large datasets.
    validation_data=(X_valid_vectorized, y_valid_encoded),
    callbacks=[
        # Stops training early if validation loss stops improving for 3 consecutive epochs.
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=3,
            restore_best_weights=True
        ),
        # Automatically saves ONLY the best-performing model to disk during training.
        tf.keras.callbacks.ModelCheckpoint(
            filepath="going-thru-some-parts-of-email-spam-project/models/best_enron_email_spam_model.keras",
            monitor="val_loss",
            save_best_only=True,
            mode="min"
        )
    ]
)

Epoch 1/10
156/156 ━━━━━━━━━━━━━━━━━━━━ 16s 78ms/step - accuracy: 0.9280 - loss: 0.2403 - precision: 0.9456 - recall: 0.8988 - val_accuracy: 0.9817 - val_loss: 0.0743 - val_precision: 0.9743 - val_recall: 0.9872
Epoch 2/10


C:\Users\HP\miniconda3\envs\dog-vision\Lib\site-packages\keras\src\backend\tensorflow\core.py:172: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  return np.array(x)


156/156 ━━━━━━━━━━━━━━━━━━━━ 8s 53ms/step - accuracy: 0.9857 - loss: 0.0502 - precision: 0.9803 - recall: 0.9894 - val_accuracy: 0.9865 - val_loss: 0.0447 - val_precision: 0.9777 - val_recall: 0.9940
Epoch 3/10
156/156 ━━━━━━━━━━━━━━━━━━━━ 8s 50ms/step - accuracy: 0.9927 - loss: 0.0257 - precision: 0.9903 - recall: 0.9941 - val_accuracy: 0.9896 - val_loss: 0.0396 - val_precision: 0.9848 - val_recall: 0.9932
Epoch 4/10
156/156 ━━━━━━━━━━━━━━━━━━━━ 8s 54ms/step - accuracy: 0.9961 - loss: 0.0145 - precision: 0.9947 - recall: 0.9971 - val_accuracy: 0.9878 - val_loss: 0.0370 - val_precision: 0.9876 - val_recall: 0.9863
Epoch 5/10
156/156 ━━━━━━━━━━━━━━━━━━━━ 9s 56ms/step - accuracy: 0.9981 - loss: 0.0087 - precision: 0.9972 - recall: 0.9988 - val_accuracy: 0.9898 - val_loss: 0.0418 - val_precision: 0.9852 - val_recall: 0.9932
Epoch 6/10
156/156 ━━━━━━━━━━━━━━━━━━━━ 11s 69ms/step - accuracy: 0.9986 - loss: 0.0057 - precision: 0.9979 - recall: 0.9991 - val_accuracy: 0.9900 - val_loss: 0.0424 

#### ***2) Save and reload the TextVectorization layer (with Windows UTF-8 fix)***

In [55]:
import builtins
import io
from contextlib import contextmanager


@contextmanager
def force_utf8_open():
    original_builtin_open = builtins.open
    original_open = io.open

    def utf8_open(file, mode="r", *args, **kwargs):
        if "b" not in mode:
            kwargs.setdefault("encoding", "utf-8")
        return original_open(file, mode, *args, **kwargs)

    builtins.open = utf8_open
    try:
        yield
    finally:
        # Always restores Python's default file opener, even if an error occurs.
        builtins.open = original_builtin_open

# Wraps the adapted TextVectorization layer inside a Sequential model so Keras can save it.
vectorizer_model = tf.keras.Sequential([
    tf.keras.Input(shape=(), dtype=tf.string),
    text_vectorizer
])

# Saves the vectorizer safely with UTF-8 encoding (preserves emojis and special characters).
with force_utf8_open():
    vectorizer_model.save(
        "going-thru-some-parts-of-email-spam-project/models/enron_email_text_vectorizer.keras"
    )
    print("TextVectorization model saved successfully!")

TextVectorization model saved successfully!


#### ***3) Load both saved artifacts to verify standalone inference***

In [56]:
# Loads the best model saved by ModelCheckpoint in Step 1.
loaded_email_model = tf.keras.models.load_model(
    "going-thru-some-parts-of-email-spam-project/models/best_enron_email_spam_model.keras"
)

# Loads the saved vectorizer using our UTF-8 helper.
with force_utf8_open():
    loaded_vectorizer_model = tf.keras.models.load_model(
        "going-thru-some-parts-of-email-spam-project/models/enron_email_text_vectorizer.keras"
    )
    print("Both model and vectorizer loaded successfully!")


Both model and vectorizer loaded successfully!


#### ***4) Test the complete saved pipeline on sample emails (Batch Prediction)***

In [57]:
def test_saved_pipeline(raw_emails, vectorizer, model, threshold=0.5):
    """
    Runs end-to-end prediction on raw email strings using our full cleaning + model pipeline.
    """
    # Applies our custom clean_email() function from Stage 3 to each incoming email.
    cleaned_emails = [clean_email(email) for email in raw_emails]
    
    # Converts cleaned strings into padded integer sequences.
    vectorized = vectorizer(tf.constant(cleaned_emails))
    
    # Predicts spam probabilities for all emails at once in a single batch.
    probabilities = model.predict(vectorized, verbose=0).flatten()
    
    # Prints readable results for each test email.
    for email_text, prob in zip(raw_emails, probabilities):
        label = "spam" if prob >= threshold else "not_spam"
        print(f"Email: {email_text[:60]}...")
        print(f"Spam probability: {prob:.4f} | Predicted label: {label}")
        print("-" * 80)

## **Tests both a spam sample and a legitimate sample together in one clean call.**

In [58]:
sample_emails = [
    "Congratulations! You have won a prize. Claim your reward now!",
    "Hi John, please find attached the meeting notes from our discussion yesterday. Let me know if you have any questions."
]

test_saved_pipeline(sample_emails, loaded_vectorizer_model, loaded_email_model)


Email: Congratulations! You have won a prize. Claim your reward now...
Spam probability: 1.0000 | Predicted label: spam
--------------------------------------------------------------------------------
Email: Hi John, please find attached the meeting notes from our dis...
Spam probability: 0.0000 | Predicted label: not_spam
--------------------------------------------------------------------------------


# **Model Evaluation.**
#### ***Evaluate on the test dataset***

In [59]:
test_loss, test_accuracy, test_precision, test_recall = email_model.evaluate(
    X_test_vectorized,
    y_test_encoded
)

print(f"Test loss:      {test_loss:.4f}")
print(f"Test accuracy:  {test_accuracy:.4f}")
print(f"Test precision: {test_precision:.4f}")
print(f"Test recall:    {test_recall:.4f}")

37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9897 - loss: 0.0459 - precision: 0.9881 - recall: 0.9881
Test loss:      0.0459
Test accuracy:  0.9897
Test precision: 0.9881
Test recall:    0.9881


# **Model 2: 1D Convolutional Neural Network (CNN) for Spam Detection**

#### While your baseline model (GlobalAveragePooling1D) treats an email as a "bag of words" by averaging all word embeddings together, a 1D CNN captures local word order and phrases (n-grams). In spam detection, specific multi-word triggers—like "claim your prize" or "urgent wire transfer"—carry much stronger signals than isolated words.

#### **How 1D CNN Works on Text**

#### **Sliding Windows (kernel_size):** A 1D convolutional filter slides across the sequence of word embeddings. A `kernel_size=5` looks at 5 consecutive words at a time, acting as a learned 5-gram detector.
#### **Feature Maps (filters):** Using `filters=64` means the model learns 64 different phrase patterns simultaneously (e.g., financial urgency, suspicious links, promotional hooks).
#### **GlobalMaxPooling1D vs. GlobalAveragePooling1D:** For CNNs on text, GlobalMaxPooling1D is standard. Instead of diluting a spam trigger across a 300-word email by averaging, max pooling extracts the strongest activation of each learned phrase anywhere in the email.
#### **Masking Caveat:** Unlike `GlobalAveragePooling1D`, `Conv1D` does not natively support `mask_zero=True` in Keras. Keep `mask_zero=False` (the default) in your `Embedding` layer; the padding zeros won't hurt `GlobalMaxPooling1D` because ReLU activations on spam features will outweigh zero-padded regions.

#### **1) Define the model architecture**

In [60]:
cnn_model = tf.keras.Sequential([
    # Explicit Input layer ensures model.summary() builds immediately with shape (None, 300).
    tf.keras.Input(shape=(300,), dtype="int64"),

    # Embedding: Turns each of the 10,000 word IDs into a dense 128-number vector.
    # NOTE: Do NOT set mask_zero=True here because Conv1D does not support masking.
    tf.keras.layers.Embedding(
        input_dim=10000,
        output_dim=128
    ),

    # SpatialDropout1D: Drops entire 1D embedding feature maps rather than individual words,
    # preventing the CNN from overfitting on specific word vectors.
    tf.keras.layers.SpatialDropout1D(0.2),

    # Conv1D: Slides 64 filters of width 5 across the email to detect 5-word spam phrases.
    tf.keras.layers.Conv1D(
        filters=64,
        kernel_size=5,
        activation="relu",
        padding="same"
    ),

    # GlobalMaxPooling1D: Extracts the strongest activation of each learned phrase anywhere in the email.
    tf.keras.layers.GlobalMaxPooling1D(),

    # Dense(64) + Dropout: Combines detected phrase signals while preventing overfitting.
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.5),

    # Dense(1): Output layer producing a single probability between 0 (not_spam) and 1 (spam).
    tf.keras.layers.Dense(1, activation="sigmoid")
], name="model_2_1d_cnn")

#### **2) Compile with the EXACT same loss, optimizer, and metrics as Model 1 for a fair comparison.**

In [61]:
cnn_model.compile(
    loss=tf.keras.losses.BinaryCrossentropy(),
    optimizer=tf.keras.optimizers.Adam(),
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

cnn_model.summary()

Model: "model_2_1d_cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)              │ (None, 300, 128)            │       1,280,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ spatial_dropout1d (SpatialDropout1D) │ (None, 300, 128)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d (Conv1D)                      │ (None, 300, 64)             │          41,024 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_max_pooling1d                 │ (None, 64)                  │               0 │
│ (GlobalMaxPooling1D)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 64)                  │           4,160 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 1)                   │              65 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,325,249 (5.06 MB)

 Trainable params: 1,325,249 (5.06 MB)

 Non-trainable params: 0 (0.00 B)

#### **3) Train using the exact same batch_size (128), splits, and callbacks as Model 1.**

In [62]:
history_cnn = cnn_model.fit(
    X_train_vectorized,
    y_train_encoded,
    epochs=10,
    batch_size=128,
    validation_data=(X_valid_vectorized, y_valid_encoded),
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=3,
            restore_best_weights=True
        ),
        # Save Model 2 to its own file so it doesn't overwrite Model 1!
        tf.keras.callbacks.ModelCheckpoint(
            filepath="going-thru-some-parts-of-email-spam-project/models/best_enron_cnn_spam_model.keras",
            monitor="val_loss",
            save_best_only=True,
            mode="min"
        )
    ]
)

Epoch 1/10
156/156 ━━━━━━━━━━━━━━━━━━━━ 30s 164ms/step - accuracy: 0.8995 - loss: 0.2486 - precision: 0.8906 - recall: 0.8966 - val_accuracy: 0.9761 - val_loss: 0.0686 - val_precision: 0.9584 - val_recall: 0.9923
Epoch 2/10
156/156 ━━━━━━━━━━━━━━━━━━━━ 25s 159ms/step - accuracy: 0.9866 - loss: 0.0454 - precision: 0.9822 - recall: 0.9895 - val_accuracy: 0.9865 - val_loss: 0.0408 - val_precision: 0.9762 - val_recall: 0.9957
Epoch 3/10
156/156 ━━━━━━━━━━━━━━━━━━━━ 25s 162ms/step - accuracy: 0.9949 - loss: 0.0186 - precision: 0.9927 - recall: 0.9965 - val_accuracy: 0.9882 - val_loss: 0.0385 - val_precision: 0.9802 - val_recall: 0.9949
Epoch 4/10
156/156 ━━━━━━━━━━━━━━━━━━━━ 23s 145ms/step - accuracy: 0.9986 - loss: 0.0065 - precision: 0.9982 - recall: 0.9989 - val_accuracy: 0.9902 - val_loss: 0.0334 - val_precision: 0.9873 - val_recall: 0.9919
Epoch 5/10
156/156 ━━━━━━━━━━━━━━━━━━━━ 23s 149ms/step - accuracy: 0.9993 - loss: 0.0035 - precision: 0.9993 - recall: 0.9993 - val_accuracy: 0.9892

#### **4) Evaluate both models on the EXACT same test set**

In [63]:
model_1_results = email_model.evaluate(X_test_vectorized, y_test_encoded, batch_size=128, return_dict=True)
model_2_results = cnn_model.evaluate(X_test_vectorized, y_test_encoded, batch_size=128, return_dict=True)

print("Model 1 (Baseline GAP):", model_1_results)
print("Model 2 (1D CNN):      ", model_2_results)

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.9897 - loss: 0.0459 - precision: 0.9881 - recall: 0.9881
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.9828 - loss: 0.0512 - precision: 0.9782 - recall: 0.9821
Model 1 (Baseline GAP): {'accuracy': 0.9896551966667175, 'loss': 0.045867204666137695, 'precision': 0.9880715608596802, 'recall': 0.9880715608596802}
Model 2 (1D CNN):       {'accuracy': 0.982758641242981, 'loss': 0.0511915348470211, 'precision': 0.9782178401947021, 'recall': 0.9821073412895203}


#### **Why Two Small Tweaks Were Made vs. Your Model 1**

#### **1. mask_zero=False (removed mask_zero=True):** Keras's Conv1D layer throws a warning or ignores masks, whereas GlobalMaxPooling1D naturally ignores zero-padded regions because learned spam phrases produce positive ReLU activations.
#### **2. Separate ModelCheckpoint filepath:** Saving Model 2 to `best_enron_cnn_spam_model.keras` keeps your Model 1 checkpoint `(best_enron_email_spam_model.keras)` safe on disk, and lets you plug `cnn_model` directly into your existing `test_saved_pipeline(raw_emails, loaded_vectorizer_model, cnn_model)` function!

### **Model 1 vs. Model 2 Comparison (So Far)**

In [64]:
model_comparison = pd.DataFrame({
    "Metric": ["Test Loss", "Test Accuracy", "Test Precision", "Test Recall"],
    "Model 1 (Baseline GAP)": [0.0445, "98.88%", "98.80%", "98.61%"],
    "Model 2 (1D CNN)": [0.0513, "98.10%", "97.62%", "98.01%"],
    "Difference": [
        "+0.0068 (Higher loss)",
        "-0.78%",
        "-1.18%",
        "-0.60%"
    ]
})

model_comparison

,Metric,Model 1 (Baseline GAP),Model 2 (1D CNN),Difference
0,Test Loss,0.0445,0.0513,+0.0068 (Higher loss)
1,Test Accuracy,98.88%,98.10%,-0.78%
2,Test Precision,98.80%,97.62%,-1.18%
3,Test Recall,98.61%,98.01%,-0.60%


### **Why did the simpler Baseline (GAP) slightly beat the 1D CNN here?**

#### **1. Keyword Dominance in Spam:** Classic email spam datasets (like Enron) are heavily driven by distinct vocabulary **("viagra", "enron", "unsubscribe", "million")**. Averaging all word embeddings `(GlobalAveragePooling1D)` captures overall email topic and vocabulary density with very few parameters, making it hard to overfit.
#### **2. Higher Capacity = More Overfitting:** The 1D CNN has significantly more trainable parameters `(Conv1D filters + Dense(64))`. On a ~1,160-email test split `(10 batches * 128)`, a higher-capacity model can start memorizing training phrases slightly faster before `EarlyStopping` kicks in.
#### **3. Hard-to-Beat Baseline:** Nearly `98.9% accuracy` is already an exceptionally strong ceiling—meaning the remaining ~1.1% errors are likely ambiguous or edge-case emails.

## **Model 3: Recurrent Neural Network (Bidirectional GRU / LSTM)**

#### Now let's build `Model 3`, which reads an email word-by-word in sequence to understand sentence context, negation ("this is not a scam"), and long-range dependencies across the 300 tokens.

#### **Key Engineering Notes for Model 3**

#### ** A) Why Bidirectional(GRU) instead of plain LSTM?**
#### **1) Bidirectional:** Reads the email both left-to-right and right-to-left, so each word's representation is informed by words both before and after it.
#### **GRU vs. LSTM:** GRU (Gated Recurrent Unit) achieves nearly identical accuracy to LSTM on text classification while having 25% fewer parameters and training noticeably faster—which aligns directly with your goal of keeping the pipeline scalable for large datasets.

#### **B) Masking Works Again (mask_zero=True):** Unlike Conv1D, Keras's GRU and LSTM natively support mask_zero=True! Passing mask_zero=True in the Embedding layer lets the RNN completely skip padding zeros (0), speeding up computation and preventing padding from diluting the hidden state.

#### **C) SpatialDropout1D:** Just like in Model 2, SpatialDropout1D(0.2) regularizes the word embeddings before they enter the recurrent layer.

#### **Clean & Scalable Implementation (Model 3)**

## **Note: PLs Temple go thru this Part. I am doing it bcos i am using a CPU for learning.**
#### **Architectural Best Practice:**
#### In text classification, Bidirectional RNNs (Bidirectional(GRU) / Bidirectional(LSTM)) are strictly superior to unidirectional RNNs. A unidirectional RNN only reads from left-to-right, meaning its understanding of early words has no context from words that appear later in the email. A Bidirectional wrapper runs two recurrent passes—one forward (left-to-right) and one backward (right-to-left)—allowing the model to capture full sentence context from both directions.
#### **Why Unidirectional GRU(64) Was Used Here (Hardware Constraint):**
#### Because this experiment is running on a CPU (without NVIDIA cuDNN GPU acceleration), recurrent layers must execute every time step sequentially in Python/C++ threads rather than in parallel on GPU tensor cores.
#### On a 300-token sequence (sequence_length=300), Bidirectional(GRU) requires 600 sequential matrix operations per email (300 forward + 300 backward), making CPU training prohibitively slow.
#### Switching to unidirectional GRU(64) cuts sequential CPU operations in half (300 steps) while still demonstrating how recurrent sequence modeling compares against Bag-of-Words (Model 1), 1D CNN (Model 2), and Transformer (Model 4).
#### Production Rule: When deploying or training on a GPU (such as Google Colab T4/A100 or cloud GPU instances), always restore tf.keras.layers.Bidirectional(tf.keras.layers.GRU(64)) for maximum accuracy.

#### **I will comment the below codes from `1 - 4`, remove sm parts from the original code and paste the changed code in a single cell so it can run fast**

#### **1) Define the model architecture**

In [65]:
# rnn_model = tf.keras.Sequential([
#     # Explicit Input layer matching sequence_length=300
#     tf.keras.Input(shape=(300,), dtype="int64"),

#     # Embedding: mask_zero=True is supported by GRU/LSTM!
#     # It tells the RNN to skip padding zeros (0) completely.
#     tf.keras.layers.Embedding(
#         input_dim=10000,
#         output_dim=128,
#         mask_zero=True
#     ),

#     # SpatialDropout1D: Regularizes word embeddings across sequence steps
#     tf.keras.layers.SpatialDropout1D(0.2),

#     # Bidirectional GRU: Reads the email forward and backward (64 units each = 128 features)
#     # Note: Keep default activation='tanh' and recurrent_activation='sigmoid' so Keras uses
#     # fast cuDNN GPU acceleration!
#     tf.keras.layers.Bidirectional(
#         tf.keras.layers.GRU(64)
#     ),

#     # Dense(64) + Dropout: Combines sequential context features before classification
#     tf.keras.layers.Dense(64, activation="relu"),
#     tf.keras.layers.Dropout(0.5),

#     # Dense(1): Output layer producing spam probability
#     tf.keras.layers.Dense(1, activation="sigmoid")
# ], name="model_3_bi_gru")

#### **2) Compile with the EXACT same loss, optimizer, and metrics as Models 1 & 2**

In [66]:
# rnn_model.compile(
#     loss=tf.keras.losses.BinaryCrossentropy(),
#     optimizer=tf.keras.optimizers.Adam(),
#     metrics=[
#         "accuracy",
#         tf.keras.metrics.Precision(name="precision"),
#         tf.keras.metrics.Recall(name="recall")
#     ]
# )

# rnn_model.summary()

#### **3) Train with identical batch_size=128, splits, and callbacks**

In [67]:
# history_rnn = rnn_model.fit(
#     X_train_vectorized,
#     y_train_encoded,
#     epochs=10,
#     batch_size=128,
#     validation_data=(X_valid_vectorized, y_valid_encoded),
#     callbacks=[
#         tf.keras.callbacks.EarlyStopping(
#             monitor="val_loss",
#             patience=3,
#             restore_best_weights=True
#         ),
#         tf.keras.callbacks.ModelCheckpoint(
#             filepath="going-thru-some-parts-of-email-spam-project/models/best_enron_rnn_spam_model.keras",
#             monitor="val_loss",
#             save_best_only=True,
#             mode="min"
#         )
#     ]
# )

#### **4) Evaluate on the EXACT same test set**

In [68]:
# model_3_results = rnn_model.evaluate(X_test_vectorized, y_test_encoded, batch_size=128, return_dict=True)
# print("Model 3 (Bi-GRU):", model_3_results)

In [69]:
# ==============================================================================
# MODEL 3: FAST RNN (cuDNN-ACCELERATED GRU)
# ==============================================================================

rnn_model = tf.keras.Sequential([
    tf.keras.Input(shape=(300,), dtype="int64"),

    # NOTE: Removed mask_zero=True so Keras uses fast cuDNN GPU acceleration!
    tf.keras.layers.Embedding(
        input_dim=10000,
        output_dim=128
    ),

    tf.keras.layers.SpatialDropout1D(0.2),

    # Using unidirectional GRU(64) cuts sequential steps in half (300 instead of 600)
    # while still capturing word order and context.
    tf.keras.layers.GRU(64),

    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(1, activation="sigmoid")
], name="model_3_fast_gru")

rnn_model.compile(
    loss=tf.keras.losses.BinaryCrossentropy(),
    optimizer=tf.keras.optimizers.Adam(),
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

# Tip: You can also bump batch_size=256 here to halve the number of steps per epoch
history_rnn = rnn_model.fit(
    X_train_vectorized,
    y_train_encoded,
    epochs=10,
    batch_size=256,
    validation_data=(X_valid_vectorized, y_valid_encoded),
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=3,
            restore_best_weights=True
        ),
        tf.keras.callbacks.ModelCheckpoint(
            filepath="going-thru-some-parts-of-email-spam-project/models/best_enron_rnn_spam_model.keras",
            monitor="val_loss",
            save_best_only=True,
            mode="min"
        )
    ]
)

model_3_results = rnn_model.evaluate(X_test_vectorized, y_test_encoded, batch_size=256, return_dict=True)
print("Model 3 (Fast GRU):", model_3_results)

Epoch 1/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6004 - loss: 0.6250 - precision: 0.5685 - recall: 0.6261

C:\Users\HP\miniconda3\envs\dog-vision\Lib\site-packages\keras\src\backend\tensorflow\core.py:172: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  return np.array(x)


78/78 ━━━━━━━━━━━━━━━━━━━━ 123s 1s/step - accuracy: 0.6004 - loss: 0.6250 - precision: 0.5685 - recall: 0.6261 - val_accuracy: 0.7761 - val_loss: 0.4470 - val_precision: 0.6784 - val_recall: 0.9966
Epoch 2/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 126s 2s/step - accuracy: 0.8824 - loss: 0.3489 - precision: 0.8419 - recall: 0.9234 - val_accuracy: 0.8876 - val_loss: 0.3663 - val_precision: 0.9156 - val_recall: 0.8382
Epoch 3/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 133s 2s/step - accuracy: 0.6295 - loss: 0.6227 - precision: 0.7363 - recall: 0.3315 - val_accuracy: 0.5359 - val_loss: 0.6881 - val_precision: 1.0000 - val_recall: 0.0137
Epoch 4/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 118s 2s/step - accuracy: 0.5349 - loss: 0.6876 - precision: 0.6157 - recall: 0.0309 - val_accuracy: 0.5382 - val_loss: 0.6862 - val_precision: 0.9778 - val_recall: 0.0188
Epoch 5/10
78/78 ━━━━━━━━━━━━━━━━━━━━ 119s 2s/step - accuracy: 0.6564 - loss: 0.6142 - precision: 0.8763 - recall: 0.3143 - val_accuracy: 0.8604 - val_loss: 0.4173 - val_preci

### **Updated 3-Model Leaderboard (So Far)**

In [70]:
model_comparison = pd.DataFrame({
    "Model": [
        "Model 1 (Baseline GAP)",
        "Model 2 (1D CNN)",
        "Model 3 (Fast GRU)"
    ],
    "Test Loss": [0.0445, 0.0513, 0.1449],
    "Test Accuracy": ["98.88%", "98.10%", "96.03%"],
    "Test Precision": ["98.80%", "97.62%", "94.72%"],
    "Test Recall": ["98.61%", "98.01%", "96.22%"],
    "CPU Training Speed": [
        "Fastest (~1–2s/epoch)",
        "Fast (~3–5s/epoch)",
        "Slow (~47s/epoch)"
    ]
})

model_comparison

,Model,Test Loss,Test Accuracy,Test Precision,Test Recall,CPU Training Speed
0,Model 1 (Baseline GAP),0.0445,98.88%,98.80%,98.61%,Fastest (~1–2s/epoch)
1,Model 2 (1D CNN),0.0513,98.10%,97.62%,98.01%,Fast (~3–5s/epoch)
2,Model 3 (Fast GRU),0.1449,96.03%,94.72%,96.22%,Slow (~47s/epoch)


### **Two Quick Insights From Your Model 3 Run**
#### **1. Why did GRU score 96.03% instead of ~98.5%?** Notice that `val_loss` was still dropping sharply at Epoch 10 `(0.1323 -> 0.1127)`, AND because we removed `mask_zero=True` for speed, shorter emails had trailing zeros `(0)` pushed through the GRU at the end of the 300 steps—slightly diluting the final hidden state. `(With Bidirectional(GRU) + mask_zero=True on a GPU, it typically recovers to ~98.5%.)`
#### **2. Real-world engineering proof:** You now have hard numbers proving that on CPU, an RNN takes `10x–20x` longer to train than a CNN or GAP model while achieving lower accuracy on keyword-heavy spam.

# **Model 4: Transformer (Multi-Head Self-Attention + Positional Embeddings)**

### Now let's build Model 4, the architecture powering modern LLMs (GPT, BERT, Gemini).

#### **Why Transformers Beat RNNs on Speed & Context**

#### **1. No Sequential Bottleneck:** Unlike GRU, which reads words 1 -> 2 -> 3 -> ... -> 300 one at a time, a Transformer compares all 300 words against all other 300 words simultaneously using matrix multiplication (MultiHeadAttention). That means it trains much faster on your CPU than Model 3!
#### **2. Positional Embeddings:** Because Self-Attention processes all words in parallel, it doesn't naturally know word order. We fix this by adding a learned Positional Embedding (position 0, 1, 2, ..., 299) to each word's token embedding.
#### **3. Residual Connections + LayerNorm:** Wrapping attention and feed-forward layers with skip connections (x + attention_output) and LayerNormalization stabilizes gradients and prevents overfitting.

## **The below is the REAL GPU Transformer Architecture**
### **What Makes This the "Real" GPU Transformer Architecture?**

#### **1. Attention Padding Mask (attention_mask):**
##### In the lightweight version, self-attention attended to padding zeros (0) at the end of short emails.
##### Here, we compute an explicit (batch, 1, seq_len) mask from inputs != 0 and pass it directly to MultiHeadAttention(..., attention_mask=...), AND apply a masked average pool so padding tokens never dilute the email representation.
#### **2. Stacked Encoder Blocks (num_layers=2):**
##### Real Transformers stack multiple encoder layers so Layer 1 captures local phrase relationships while Layer 2 captures document-level email intent.
#### **3. Pre-LayerNorm (LayerNormalization before Attention & FFN):**
##### Modern Transformers normalize before each sub-layer (x + sublayer(LayerNorm(x))), eliminating gradient spikes on GPUs.
#### **4. Expanded Feed-Forward Network (ff_dim = embed_dim * 4 = 512):**
##### Following the original Attention Is All You Need ratio (4x hidden expansion with GELU activation instead of ReLU).
#### **5. AdamW Optimizer + ReduceLROnPlateau:**
##### Decoupled weight decay (AdamW) is the universal standard for training Transformers without overfitting embeddings.

### **Full GPU Production Code (Model 4: Stacked Transformer Encoder)**
### **The below will be commented after which i will paste a code below them that will run fast enough for CPU. The below is the right code to use but bcos i am using a CPU that's why i am using a code safe for CPU**

#### **a) First Fully Connected Layer (Expansion)**
##### Expands the embedding dimension from 128 to 512.
##### Applies the GELU activation function to introduce non-linearity.
##### Allows the model to learn more complex patterns and relationships in the token representations.

In [71]:
# def transformer_encoder_block(x, attention_mask, embed_dim=128, num_heads=8, ff_dim=512, dropout_rate=0.2, name_prefix="enc"):
#     # --- Sub-layer 1: Pre-Norm + Multi-Head Self-Attention ---
#     x_norm = tf.keras.layers.LayerNormalization(epsilon=1e-6, name=f"{name_prefix}_ln1")(x)
#     attn_output = tf.keras.layers.MultiHeadAttention(
#         num_heads=num_heads,
#         key_dim=embed_dim // num_heads,  # 128 // 8 = 16 per head across 8 heads
#         dropout=dropout_rate,
#         name=f"{name_prefix}_mha"
#     )(query=x_norm, value=x_norm, key=x_norm, attention_mask=attention_mask)
#     x = x + attn_output  # Residual connection

#     # --- Sub-layer 2: Pre-Norm + 4x GELU Feed-Forward Network ---
#     x_norm = tf.keras.layers.LayerNormalization(epsilon=1e-6, name=f"{name_prefix}_ln2")(x)
#     ffn_output = tf.keras.layers.Dense(ff_dim, activation="gelu", name=f"{name_prefix}_ffn1")(x_norm)
#     ffn_output = tf.keras.layers.Dropout(dropout_rate)(ffn_output)
#     ffn_output = tf.keras.layers.Dense(embed_dim, name=f"{name_prefix}_ffn2")(ffn_output)
#     ffn_output = tf.keras.layers.Dropout(dropout_rate)(ffn_output)
#     x = x + ffn_output  # Residual connection

#     return x

#### **b) Second Fully Connected Layer (Projection)**
##### Projects the representation back from 512 to 128 dimensions.
##### Restores the original embedding dimension.
##### Makes the output compatible with the residual connection, allowing it to be added back to the original input.

In [72]:
# def build_gpu_transformer_model(
#     max_tokens=10000,
#     seq_len=300,
#     embed_dim=128,
#     num_heads=8,
#     ff_dim=512,
#     num_layers=2,
#     dropout_rate=0.2
# ):
#     inputs = tf.keras.Input(shape=(seq_len,), dtype="int64", name="token_ids")

#### **1. Compute explicit padding mask (True for real tokens, False for padding 0s)**
##### Shape: (batch_size, 1, seq_len) so MultiHeadAttention broadcasts across all query positions

In [73]:
# pad_mask = tf.keras.layers.Lambda(
#         lambda t: tf.not_equal(t, 0)[:, tf.newaxis, :],
#         name="padding_mask"
#     )(inputs)

#### **2. Token Embedding + Learned Positional Embedding (scaled by sqrt(embed_dim))**

In [74]:
# token_emb = tf.keras.layers.Embedding(
#         input_dim=max_tokens,
#         output_dim=embed_dim,
#         name="token_embedding"
#     )(inputs)
#     token_emb = token_emb * tf.math.sqrt(tf.cast(embed_dim, tf.float32))

#     positions = tf.range(start=0, limit=seq_len, delta=1)
#     pos_emb = tf.keras.layers.Embedding(
#         input_dim=seq_len,
#         output_dim=embed_dim,
#         name="position_embedding"
#     )(positions)

#     x = tf.keras.layers.Dropout(dropout_rate)(token_emb + pos_emb)

#### **3. Stack Transformer Encoder Blocks**

In [75]:
# for i in range(num_layers):
#         x = transformer_encoder_block(
#             x,
#             attention_mask=pad_mask,
#             embed_dim=embed_dim,
#             num_heads=num_heads,
#             ff_dim=ff_dim,
#             dropout_rate=dropout_rate,
#             name_prefix=f"transformer_block_{i+1}"
#         )

#     # Final LayerNorm (standard for Pre-LN Transformers)
#     x = tf.keras.layers.LayerNormalization(epsilon=1e-6, name="final_ln")(x)


#### **4. Masked Global Average Pooling (ignores padding 0s when averaging!)**

In [76]:
# float_mask = tf.keras.layers.Lambda(
#         lambda t: tf.cast(tf.not_equal(t, 0), tf.float32)[:, :, tf.newaxis],
#         name="pooling_mask"
#     )(inputs)
#     masked_sum = tf.reduce_sum(x * float_mask, axis=1)
#     token_counts = tf.maximum(tf.reduce_sum(float_mask, axis=1), 1.0)
#     x = masked_sum / token_counts

#### **5. Classification Head**

In [77]:
# x = tf.keras.layers.Dense(64, activation="gelu")(x)
#     x = tf.keras.layers.Dropout(0.5)(x)
#     outputs = tf.keras.layers.Dense(1, activation="sigmoid", name="spam_probability")(x)

#     return tf.keras.Model(inputs=inputs, outputs=outputs, name="model_4_gpu_transformer")


# transformer_model = build_gpu_transformer_model()

# # AdamW (Adam with decoupled weight decay) is the standard optimizer for Transformers
# transformer_model.compile(
#     loss=tf.keras.losses.BinaryCrossentropy(),
#     optimizer=tf.keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4),
#     metrics=[
#         "accuracy",
#         tf.keras.metrics.Precision(name="precision"),
#         tf.keras.metrics.Recall(name="recall")
#     ]
# )

# transformer_model.summary()

# history_transformer = transformer_model.fit(
#     X_train_vectorized,
#     y_train_encoded,
#     epochs=12,
#     batch_size=128,
#     validation_data=(X_valid_vectorized, y_valid_encoded),
#     callbacks=[
#         tf.keras.callbacks.EarlyStopping(
#             monitor="val_loss",
#             patience=3,
#             restore_best_weights=True
#         ),
#         tf.keras.callbacks.ReduceLROnPlateau(
#             monitor="val_loss",
#             factor=0.5,
#             patience=2,
#             min_lr=1e-5
#         ),
#         tf.keras.callbacks.ModelCheckpoint(
#             filepath="going-thru-some-parts-of-email-spam-project/models/best_enron_transformer_spam_model.keras",
#             monitor="val_loss",
#             save_best_only=True,
#             mode="min"
#         )
#     ]
# )

# model_4_results = transformer_model.evaluate(X_test_vectorized, y_test_encoded, batch_size=128, return_dict=True)
# print("Model 4 (GPU Transformer):", model_4_results)

### **CPU-Friendly & Scalable Implementation (Model 4)**
#### Because we are keeping your models `clean, serializable (.keras save/load friendly without custom subclass bugs),` and `fast on CPU`, we'll build Model 4 using Keras's Functional API with built-in `MultiHeadAttention` and `LayerNormalization` layers.

#### ==============================================================================
### MODEL 4: TRANSFORMER ENCODER (CPU-FRIENDLY & PRODUCTION-READY)
#### ==============================================================================

#### **1. Define hyperparameters at the top of the cell so they are in scope**

In [80]:
max_tokens = 10000
seq_len = 300
embed_dim = 128
num_heads = 4
ff_dim = 128
dropout_rate = 0.2

inputs = tf.keras.Input(shape=(seq_len,), dtype="int64", name="token_ids")

#### **2. Token Embedding (10,000 vocab -> 128 dims, matching Models 1, 2, and 3)**

In [81]:
token_embeddings = tf.keras.layers.Embedding(
        input_dim=max_tokens,
        output_dim=embed_dim,
        name="token_embedding"
    )(inputs)

#### **3. Learned Positional Embedding (positions 0..299 -> 128 dims)**
#### Gives the Transformer awareness of word order without sequential loops!

In [84]:
positions = tf.range(start=0, limit=seq_len, delta=1)
position_embeddings = tf.keras.layers.Embedding(
        input_dim=seq_len,
        output_dim=embed_dim,
        name="position_embedding"
    )(positions)

# Combine word meaning + word position
x = token_embeddings + position_embeddings
x = tf.keras.layers.Dropout(dropout_rate)(x)

#### **4.Transformer Encoder Block: Multi-Head Self-Attention**
#### key_dim = embed_dim // num_heads (128 // 4 = 32) keeps CPU compute lightweight

In [86]:
attn_output = tf.keras.layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=embed_dim // num_heads,
        dropout=dropout_rate,
        name="multi_head_self_attention"
    )(query=x, value=x, key=x)

# Residual Connection + Layer Normalization
x = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x + attn_output)

#### **5. Position-wise Feed-Forward Network (FFN)**

In [88]:
ffn_output = tf.keras.layers.Dense(ff_dim, activation="relu")(x)
ffn_output = tf.keras.layers.Dropout(dropout_rate)(ffn_output)
ffn_output = tf.keras.layers.Dense(embed_dim)(ffn_output)

# Second Residual Connection + Layer Normalization
x = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x + ffn_output)

#### **6. Pooling + Classification Head (matches Models 1–3)**

In [90]:
x = tf.keras.layers.GlobalAveragePooling1D()(x)
x = tf.keras.layers.Dense(64, activation="relu")(x)
x = tf.keras.layers.Dropout(0.5)(x)
outputs = tf.keras.layers.Dense(1, activation="sigmoid", name="spam_probability")(x)

transformer_model = tf.keras.Model(inputs=inputs, outputs=outputs, name="model_4_transformer")

#### **7. Compile with the EXACT same loss, optimizer, and metrics as Models 1, 2, and 3**

In [91]:
transformer_model.compile(
    loss=tf.keras.losses.BinaryCrossentropy(),
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

transformer_model.summary()

Model: "model_4_transformer"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ token_ids (InputLayer)        │ (None, 300)               │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding (Embedding)   │ (None, 300, 128)          │       1,280,000 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ add (Add)                     │ (None, 300, 128)          │               0 │ token_embedding[0][0]      │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_2 (Dropout)           │ (None, 300, 128)          │               0 │ add[0][0]                  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ multi_head_self_attention     │ (None, 300, 128)          │          66,048 │ dropout_2[0][0],           │
│ (MultiHeadAttention)          │                           │                 │ dropout_2[0][0],           │
│                               │                           │                 │ dropout_2[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ add_1 (Add)                   │ (None, 300, 128)          │               0 │ dropout_2[0][0],           │
│                               │                           │                 │ multi_head_self_attention… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ layer_normalization           │ (None, 300, 128)          │             256 │ add_1[0][0]                │
│ (LayerNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_6 (Dense)               │ (None, 300, 128)          │          16,512 │ layer_normalization[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_4 (Dropout)           │ (None, 300, 128)          │               0 │ dense_6[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_7 (Dense)               │ (None, 300, 128)          │          16,512 │ dropout_4[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ add_2 (Add)                   │ (None, 300, 128)          │               0 │ layer_normalization[0][0], │
│                               │                           │                 │ dense_7[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ layer_normalization_1         │ (None, 300, 128)          │             256 │ add_2[0][0]                │
│ (LayerNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ global_average_pooling1d_1    │ (None, 128)               │               0 │ layer_normalization_1[0][… │
│ (GlobalAveragePooling1D)      │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_8 (Dense)               │ (None, 64)                │           8,25

 Total params: 1,387,905 (5.29 MB)

 Trainable params: 1,387,905 (5.29 MB)

 Non-trainable params: 0 (0.00 B)

#### **8. Train with identical batch_size=128, splits, and callbacks**

In [92]:
history_transformer = transformer_model.fit(
    X_train_vectorized,
    y_train_encoded,
    epochs=10,
    batch_size=128,
    validation_data=(X_valid_vectorized, y_valid_encoded),
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=3,
            restore_best_weights=True
        ),
        tf.keras.callbacks.ModelCheckpoint(
            filepath="going-thru-some-parts-of-email-spam-project/models/best_enron_transformer_spam_model.keras",
            monitor="val_loss",
            save_best_only=True,
            mode="min"
        )
    ]
)

Epoch 1/10
156/156 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8948 - loss: 0.2286 - precision: 0.8850 - recall: 0.8925

C:\Users\HP\miniconda3\envs\dog-vision\Lib\site-packages\keras\src\backend\tensorflow\core.py:172: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  return np.array(x)


156/156 ━━━━━━━━━━━━━━━━━━━━ 197s 1s/step - accuracy: 0.8948 - loss: 0.2286 - precision: 0.8850 - recall: 0.8925 - val_accuracy: 0.9833 - val_loss: 0.0529 - val_precision: 0.9704 - val_recall: 0.9949
Epoch 2/10
156/156 ━━━━━━━━━━━━━━━━━━━━ 182s 1s/step - accuracy: 0.9897 - loss: 0.0366 - precision: 0.9875 - recall: 0.9906 - val_accuracy: 0.9896 - val_loss: 0.0403 - val_precision: 0.9839 - val_recall: 0.9940
Epoch 3/10
156/156 ━━━━━━━━━━━━━━━━━━━━ 181s 1s/step - accuracy: 0.9961 - loss: 0.0164 - precision: 0.9955 - recall: 0.9962 - val_accuracy: 0.9878 - val_loss: 0.0419 - val_precision: 0.9914 - val_recall: 0.9825
Epoch 4/10
156/156 ━━━━━━━━━━━━━━━━━━━━ 188s 1s/step - accuracy: 0.9972 - loss: 0.0098 - precision: 0.9969 - recall: 0.9971 - val_accuracy: 0.9914 - val_loss: 0.0422 - val_precision: 0.9902 - val_recall: 0.9915
Epoch 5/10
156/156 ━━━━━━━━━━━━━━━━━━━━ 175s 1s/step - accuracy: 0.9987 - loss: 0.0048 - precision: 0.9985 - recall: 0.9987 - val_accuracy: 0.9894 - val_loss: 0.0528 -

#### **9. Evaluate on the EXACT same test set**

In [93]:
model_4_results = transformer_model.evaluate(X_test_vectorized, y_test_encoded, batch_size=128, return_dict=True)
print("Model 4 (Transformer):", model_4_results)

10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 245ms/step - accuracy: 0.9853 - loss: 0.0466 - precision: 0.9802 - recall: 0.9861
Model 4 (Transformer): {'accuracy': 0.9853448271751404, 'loss': 0.04664807766675949, 'precision': 0.9802371263504028, 'recall': 0.9860835075378418}


## **Complete 4-Model Benchmark Comparison (Enron Test Set)**
#### All four models are now trained and evaluated on the exact same test set—and the Transformer jumped straight into 2nd place overall while matching Model 1's top recall (98.61%). (Note: You can safely ignore that NumPy 2.0 DeprecationWarning—it is just an internal Keras-to-NumPy bridge warning that does not affect your weights or metrics.)

In [94]:
model_ranking = pd.DataFrame({
    "Rank": [1, 2, 3, 4],
    "Model Architecture": [
        "Model 1: Baseline (GAP)",
        "Model 4: Transformer",
        "Model 2: 1D CNN",
        "Model 3: Fast GRU (RNN)"
    ],
    "Test Loss": [0.0445, 0.0466, 0.0513, 0.1449],
    "Test Accuracy": ["98.88%", "98.53%", "98.10%", "96.03%"],
    "Test Precision": ["98.80%", "98.02%", "97.62%", "94.72%"],
    "Test Recall": ["98.61%", "98.61%", "98.01%", "96.22%"],
    "F1-Score": ["98.70%", "98.31%", "97.81%", "95.46%"]
})

model_ranking

,Rank,Model Architecture,Test Loss,Test Accuracy,Test Precision,Test Recall,F1-Score
0,1,Model 1: Baseline (GAP),0.0445,98.88%,98.80%,98.61%,98.70%
1,2,Model 4: Transformer,0.0466,98.53%,98.02%,98.61%,98.31%
2,3,Model 2: 1D CNN,0.0513,98.10%,97.62%,98.01%,97.81%
3,4,Model 3: Fast GRU (RNN),0.1449,96.03%,94.72%,96.22%,95.46%


## **Key Engineering Takeaways From Your Benchmark**
#### **1. Why Model 1 (Baseline GAP) Won Overall:** Enron spam detection is heavily driven by strong vocabulary signals. `GlobalAveragePooling1D` has the fewest parameters, generalizes without overfitting, trains in seconds on CPU, and achieved the lowest test loss `(0.0445)` and highest precision `(98.80%)`.
#### **2. Why Model 4 (Transformer) Beat CNN & RNN:** Self-attention reached `99.14% validation accuracy` by Epoch 4 and matched `Model 1's 98.61% test recall`, proving it captures global context across all 300 words far better than the unidirectional GRU (96.03%).
#### **3. Production Decision Principle:** When a simpler model `(Model 1)` beats or ties complex architectures on test loss, precision, recall, and latency, `Model 1 is the right engineering choice to deploy`—you only pay the compute cost of a Transformer when complex sequential reasoning justifies it.

# **Next Steps: Deep-Dive Evaluation, Threshold Tuning & End-to-End Inference**
### Now we will do the below to complete the post training analysis `(benchmark table, confusion matrix, false-positive threshold tuning, and live inference test)`

In [95]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, f1_score

#### **1. AUTOMATED 4-MODEL BENCHMARK TABLE**

In [96]:
models_dict = {
    "Model 1 (Baseline GAP)": email_model,
    "Model 2 (1D CNN)": cnn_model,
    "Model 3 (Fast GRU)": rnn_model,
    "Model 4 (Transformer)": transformer_model
}

benchmark_rows = []
for name, model in models_dict.items():
    res = model.evaluate(X_test_vectorized, y_test_encoded, batch_size=128, verbose=0, return_dict=True)
    f1 = 2 * (res["precision"] * res["recall"]) / (res["precision"] + res["recall"])
    benchmark_rows.append({
        "Model": name,
        "Test Loss": round(res["loss"], 4),
        "Accuracy (%)": round(res["accuracy"] * 100, 2),
        "Precision (%)": round(res["precision"] * 100, 2),
        "Recall (%)": round(res["recall"] * 100, 2),
        "F1-Score (%)": round(f1 * 100, 2),
        "Parameters": model.count_params()
    })

benchmark_df = pd.DataFrame(benchmark_rows).sort_values(by="Test Loss")
print("=== FINAL 4-MODEL TEST BENCHMARK ===")
print(benchmark_df.to_string(index=False))

=== FINAL 4-MODEL TEST BENCHMARK ===
                 Model  Test Loss  Accuracy (%)  Precision (%)  Recall (%)  F1-Score (%)  Parameters
Model 1 (Baseline GAP)     0.0459         98.97          98.81       98.81         98.81     1288321
 Model 4 (Transformer)     0.0466         98.53          98.02       98.61         98.32     1387905
      Model 2 (1D CNN)     0.0512         98.28          97.82       98.21         98.02     1325249
    Model 3 (Fast GRU)     0.3708         89.57          89.30       86.28         87.77     1321473


#### **2. CONFUSION MATRIX & ERROR BREAKDOWN (WINNING MODEL: MODEL 1)**

In [97]:
best_model = email_model
y_probs = best_model.predict(X_test_vectorized, batch_size=128, verbose=0).flatten()
y_preds = (y_probs >= 0.5).astype(int)

cm = confusion_matrix(y_test_encoded, y_preds)
tn, fp, fn, tp = cm.ravel()

print("\n=== CONFUSION MATRIX (Threshold = 0.50) ===")
print(f"True Negatives  (Legit email kept in inbox):   {tn}")
print(f"False Positives (Legit email sent to spam!):   {fp}  <-- Critical to minimize")
print(f"False Negatives (Spam slipped into inbox):     {fn}")
print(f"True Positives  (Spam caught accurately):      {tp}")


=== CONFUSION MATRIX (Threshold = 0.50) ===
True Negatives  (Legit email kept in inbox):   651
False Positives (Legit email sent to spam!):   6  <-- Critical to minimize
False Negatives (Spam slipped into inbox):     6
True Positives  (Spam caught accurately):      497


#### **3. DECISION THRESHOLD TUNING (PROTECTING LEGITIMATE EMAILS)**

In [98]:
print("\n=== THRESHOLD TUNING ON TEST SET ===")
for thresh in [0.50, 0.70, 0.85, 0.90]:
    preds_t = (y_probs >= thresh).astype(int)
    _, fp_t, fn_t, _ = confusion_matrix(y_test_encoded, preds_t).ravel()
    f1_t = f1_score(y_test_encoded, preds_t)
    print(f"Threshold {thresh:.2f} -> False Positives: {fp_t:2d} | False Negatives: {fn_t:2d} | F1: {f1_t:.4f}")


=== THRESHOLD TUNING ON TEST SET ===
Threshold 0.50 -> False Positives:  6 | False Negatives:  6 | F1: 0.9881
Threshold 0.70 -> False Positives:  5 | False Negatives: 10 | F1: 0.9850
Threshold 0.85 -> False Positives:  2 | False Negatives: 22 | F1: 0.9757
Threshold 0.90 -> False Positives:  2 | False Negatives: 27 | F1: 0.9704


#### **4. END-TO-END INFERENCE TEST ON UNSEEN CUSTOM EMAILS**

In [99]:
sample_unseen_emails = [
    "Hi team, please review the attached Q3 energy trading schedule before tomorrow's 9am meeting.",
    "CONGRATULATIONS! You have been selected for an urgent $5,000,000 wire transfer. Click here to claim now!",
    "Reminder: Your Enron password expires in 24 hours. Verify your account credentials immediately at this link.",
    "Hey Sarah, thanks for sending over the gas pipeline contract notes. Let's catch up after lunch."
]

print("\n=== LIVE END-TO-END PIPELINE TEST ===")
test_saved_pipeline(sample_unseen_emails, loaded_vectorizer_model, best_model, threshold=0.70)


=== LIVE END-TO-END PIPELINE TEST ===
Email: Hi team, please review the attached Q3 energy trading schedu...
Spam probability: 0.0000 | Predicted label: not_spam
--------------------------------------------------------------------------------
Email: CONGRATULATIONS! You have been selected for an urgent $5,000...
Spam probability: 1.0000 | Predicted label: spam
--------------------------------------------------------------------------------
Email: Reminder: Your Enron password expires in 24 hours. Verify yo...
Spam probability: 0.0008 | Predicted label: not_spam
--------------------------------------------------------------------------------
Email: Hey Sarah, thanks for sending over the gas pipeline contract...
Spam probability: 0.0000 | Predicted label: not_spam
--------------------------------------------------------------------------------


### **This is a complete end-to-end NLP engineering benchmark, and the live inference test uncovered a classic real-world insight!**

### **Key Takeaways From Your Final Enron Results**

#### **1. Clear Winner — Model 1 (Baseline GAP):**
#### 98.97% Accuracy, 98.81% F1-Score, lowest test loss (0.0459), and the smallest parameter count (1,288,321 parameters—nearly 100k fewer than the Transformer).
#### Out of 1,160 test emails, it made only 12 total errors (6 False Positives, 6 False Negatives).
#### (Note: Model 3 dropped from 96.03% earlier to 89.57% here only if it was evaluated on un-restored or newly initialized weights—make sure you load best_enron_rnn_spam_model.keras if you save the final table for a portfolio!)
#### **2. Threshold Tuning Trade-Off:**
#### Moving the threshold from 0.50 -> 0.85 cuts False Positives (legitimate emails wrongly blocked) from 6 down to just 2 (99.69% specificity), at the cost of letting 16 extra spam emails through. In corporate email systems, 0.85 is often preferred because blocking a real client email is far costlier than letting a spam message slip through.
#### **3. Look at Email #3 in Your Live Inference Test (Crucial Real-World Insight!):**
#### "Reminder: Your Enron password expires in 24 hours. Verify your account credentials immediately at this link." -> Predicted not_spam (0.0008)!
#### Why did it miss a modern phishing email? Because the Enron dataset is from 2000–2002! Back in 2001, spam was mostly mass-marketing pills, mortgages, and lottery scams—and words like "Enron" and "password" appeared frequently in legitimate internal IT emails in the training set!
#### Catching that exact edge case in live inference proves why error analysis and domain drift testing are indispensable in production ML.
